# Create fact table

### Reading silver data

In [0]:
df=spark.sql('''select * from parquet.`abfss://silver@carprojectazurestorage.dfs.core.windows.net/carsales`''')
df.display()

Branch_ID,Dealer_ID,Model_ID,Revenue,Units_Sold,Date_ID,Day,Month,Year,BranchName,DealerName,Category,Revenue_per_unit
BR0001,DLR0001,BMW-M1,13363978,2,DT00001,1,1,2017,AC Cars Motors,AC Cars Motors,BMW,6681989.0
BR0003,DLR0228,Hon-M218,17376468,3,DT00001,10,5,2017,AC Cars Motors,Deccan Motors,Hon,5792156.0
BR0004,DLR0208,Tat-M188,9664767,3,DT00002,12,1,2017,AC Cars Motors,Wiesmann Motors,Tat,3221589.0
BR0005,DLR0188,Hyu-M158,5525304,3,DT00002,16,9,2017,AC Cars Motors,Subaru Motors,Hyu,1841768.0
BR0006,DLR0168,Ren-M128,12971088,3,DT00003,20,5,2017,AC Cars Motors,Saab Motors,Ren,4323696.0
BR0008,DLR0128,Hon-M68,7321228,1,DT00004,28,4,2017,AC Cars Motors,Messerschmitt Motors,Hon,7321228.0
BR0009,DLR0108,Cad-M38,11379294,2,DT00004,31,12,2017,AC Cars Motors,Lexus Motors,Cad,5689647.0
BR0010,DLR0088,Mer-M8,11611234,2,DT00005,4,9,2017,AC Cars Motors,"IFA (including Trabant, Wartburg, Barkas) Motors",Mer,5805617.0
BR0011,DLR0002,BMW-M2,19979446,2,DT00005,2,1,2017,Acura Motors,Acura Motors,BMW,9989723.0
BR0011,DLR0069,Vol-M256,14181510,3,DT00006,9,5,2017,Acura Motors,Geo Motors,Vol,4727170.0


### reading all the dimensions

In [0]:
df_branch=spark.sql('select * from cars_catalog.gold.dim_branch')
df_branch.display()

Branch_ID,BranchName,dim_branch_key
BR0131,Audi Motors,13
BR0760,Healey Motors,41
BR0789,Hillman Motors,42
BR0938,Isotta Fraschini Motors,52
BR1040,Lada Motors,57
BR1693,Saleen Motors,1003
BR1792,Simca do Brasil Motors,86
BR1799,Simca do Brasil Motors,87
BR1955,Toyota Motors,95
BR1978,Turner Motors,1012


In [0]:
df_dealer=spark.sql('select * from cars_catalog.gold.dim_dealer')
df_dealer.display()

Dealer_ID,DealerName,dim_dealer_key
DLR0058,Fiat do Brasil Motors,145
DLR0107,Land Rover Motors,159
DLR0129,Mia Motors,166
DLR0111,Lotus Motors,176
DLR0085,Humber Motors,183
DLR0001,AC Cars Motors,51
DLR0218,Lagonda Motors,186
DLR0082,Honda Motors,55
DLR0063,Ford do Brasil Motors,64
DLR0193,Tazzari Motors,73


In [0]:
df_model=spark.sql('select * from cars_catalog.gold.dim_model')
df_model.display()

Model_ID,Category,dim_model_key
Mah-M167,Mah,185
Che-M47,Che,221
Toy-M205,Toy,85
BMW-M249,BMW,226
Mer-M122,Mer,237
Hon-M215,Hon,118
Nis-M82,Nis,145
Toy-M206,Toy,15
Mar-M139,Mar,34
Ren-M207,Ren,201


In [0]:
df_date=spark.sql('select * from cars_catalog.gold.dim_date')
df_date.display()

Date_ID,dim_date_key
DT00029,1
DT00140,9
DT00192,12
DT00444,30
DT00475,32
DT00947,53
DT00976,640
DT01028,55
DT01099,646
DT00657,686


In [0]:
df_fact=df.join(df_model,['Model_ID']).join(df_branch,['Branch_ID']).join(df_dealer,['Dealer_ID']).join(df_date,['Date_ID']).select(df['Revenue'],df['Units_Sold'],df['Revenue_per_unit'],df_model['dim_model_key'],df_branch['dim_branch_key'],df_dealer['dim_dealer_key'],df_date['dim_date_key'])
df_fact.display()


Date_ID,Dealer_ID,Branch_ID,Model_ID,Revenue,Units_Sold,Day,Month,Year,BranchName,DealerName,Category,Revenue_per_unit,Category,dim_model_key,BranchName,dim_branch_key,DealerName,dim_dealer_key,dim_date_key
DT00001,DLR0001,BR0001,BMW-M1,13363978,2,1,1,2017,AC Cars Motors,AC Cars Motors,BMW,6681989.0,BMW,195,AC Cars Motors,814,AC Cars Motors,51,1001
DT00001,DLR0228,BR0003,Hon-M218,17376468,3,10,5,2017,AC Cars Motors,Deccan Motors,Hon,5792156.0,Hon,261,AC Cars Motors,815,Deccan Motors,131,1001
DT00002,DLR0208,BR0004,Tat-M188,9664767,3,12,1,2017,AC Cars Motors,Wiesmann Motors,Tat,3221589.0,Tat,148,AC Cars Motors,1,Wiesmann Motors,67,867
DT00002,DLR0188,BR0005,Hyu-M158,5525304,3,16,9,2017,AC Cars Motors,Subaru Motors,Hyu,1841768.0,Hyu,87,AC Cars Motors,357,Subaru Motors,52,867
DT00003,DLR0168,BR0006,Ren-M128,12971088,3,20,5,2017,AC Cars Motors,Saab Motors,Ren,4323696.0,Ren,20,AC Cars Motors,683,Saab Motors,132,69
DT00004,DLR0128,BR0008,Hon-M68,7321228,1,28,4,2017,AC Cars Motors,Messerschmitt Motors,Hon,7321228.0,Hon,242,AC Cars Motors,358,Messerschmitt Motors,36,285
DT00004,DLR0108,BR0009,Cad-M38,11379294,2,31,12,2017,AC Cars Motors,Lexus Motors,Cad,5689647.0,Cad,160,AC Cars Motors,1492,Lexus Motors,82,285
DT00005,DLR0088,BR0010,Mer-M8,11611234,2,4,9,2017,AC Cars Motors,"IFA (including Trabant, Wartburg, Barkas) Motors",Mer,5805617.0,Mer,179,AC Cars Motors,933,"IFA (including Trabant, Wartburg, Barkas) Motors",119,1069
DT00005,DLR0002,BR0011,BMW-M2,19979446,2,2,1,2017,Acura Motors,Acura Motors,BMW,9989723.0,BMW,262,Acura Motors,1493,Acura Motors,239,1069
DT00006,DLR0069,BR0011,Vol-M256,14181510,3,9,5,2017,Acura Motors,Geo Motors,Vol,4727170.0,Vol,209,Acura Motors,1493,Geo Motors,1,1002


### writing fact table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.factsales'):
    delta_table=DeltaTable.forPath(spark,"abfss://gold@carprojectazurestorage.dfs.core.windows.net/factsales")
    delta_table.alias('t')\
            .merge(df_fact.alias('s'),'t.dim_branch_key=s.dim_branch_key and t.dim_dealer_key=s.dim_dealer_key and t.dim_model_key=s.dim_model_key and t.dim_date_key=s.dim_date_key')\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df_fact.write.format('Delta')\
            .mode("overwrite")\
            .option('path','abfss://gold@carprojectazurestorage.dfs.core.windows.net/factsales')\
            .saveAsTable('cars_catalog.gold.factsales')

In [0]:
df_fact_d=spark.sql('select * from cars_catalog.gold.factsales')
df_fact_d.display()

Revenue,Units_Sold,Revenue_per_unit,dim_model_key,dim_dealer_key,dim_branch_key,dim_date_key
13363978,2,6681989.0,195,51,814,1001
17376468,3,5792156.0,261,131,815,1001
9664767,3,3221589.0,148,67,1,867
5525304,3,1841768.0,87,52,357,867
12971088,3,4323696.0,20,132,683,69
7321228,1,7321228.0,242,36,358,285
11379294,2,5689647.0,160,82,1492,285
11611234,2,5805617.0,179,119,933,1069
19979446,2,9989723.0,262,239,1493,1069
14181510,3,4727170.0,209,1,1493,1002
